# A5 — Sequence Modeling with RNNs

**Goal:** Apply basic and advanced **RNN / LSTM / GRU, Encoder-Decoder** architectures to a sequence-based sentiment classification task.

---

## Dataset (Required)

### IMDB Movie Reviews (Binary Sentiment Classification)
**Access method (required):**
```python
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = ...
```

You must create a validation split from training data and pad/truncate sequences to a fixed length.

---
## What you will implement

You will implement and compare the following **sequence models**:

- **Vanilla RNN**
- **LSTM**
- **GRU**
- **Stacked LSTM + Dropout**
- **Encoder–Decoder LSTM**
- **LSTM + GRU Hybrid**

---

## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [1]:
# ============================================================
# Q0) Environment Setup
# ============================================================

import os
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


TensorFlow: 2.19.0


---
## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q9)** and continues with the **Markdown responses (Q10–Q13)**.  
These correspond to the questions listed in the assignment description on **Canvas**. Follow the instructions provided in the **preceding Markdown cells** for each step.

### Tasks

This assignment focuses on **sequence modeling for text classification** using recurrent neural networks.

You will:

- Train and evaluate the following **sequence models**:
  - **Vanilla RNN**
  - **LSTM**
  - **GRU**

- Implement additional **advanced architectures**:
  - **Stacked LSTM + Dropout**
  - **Encoder–Decoder LSTM**
  - **LSTM + GRU Hybrid**

- Use the **IMDB Movie Reviews dataset** for **binary sentiment classification**.

- Perform a **comparative analysis** of the models, including:
  - training convergence behavior
  - validation and test performance
  - explanation of architectural differences across models

Ensure that all models are **computationally feasible** to train on **CPU-only environments** by using the recommended hyperparameters unless you have access to a GPU (e.g. Google Colab).

---

## Q1 — Load Dataset & Inspect

Use the **IMDB Movie Reviews dataset** from Keras and inspect its basic structure.

### Student Tasks

- Load the IMDB dataset using `tensorflow.keras.datasets.imdb` with a vocabulary size of **10,000 words** by frequency.

- Split the training data into **training** and **validation** sets.

- Inspect the dataset by:
  - Printing the **number of training and test samples**
  - Displaying the **label distribution (train)**
  - Printing one example **Sequence length (train)**

---

In [2]:
# ============================================================
# Question Q1 — Load IMDB Dataset (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Load the IMDB Movie Reviews dataset
# 2) Print the number of training and test examples
# 3) Check the label distribution in the training set
# 4) Inspect sequence length statistics
# ============================================================

import numpy as np
from tensorflow.keras.datasets import imdb

# TODO 1: Define vocabulary size
VOCAB_SIZE = 10000

# TODO 2: Load the IMDB dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# TODO 3: Print dataset sizes
print("Train examples:", len(x_train))
print("Test examples: ", len(x_test))

# TODO 4: Print label distribution
print("Label distribution (train):", np.bincount(y_train))

# TODO 5: Compute sequence length statistics
train_lengths = np.array([len(s) for s in x_train])  # s is just a temoporary variable for each review sequence in x_train.

print(
    "Sequence length (train): min/median/max =",
    train_lengths.min(),
    int(np.median(train_lengths)),
    train_lengths.max()
)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train examples: 25000
Test examples:  25000
Label distribution (train): [12500 12500]
Sequence length (train): min/median/max = 11 178 2494


---

## Q2 — Validation Split & Sequence Padding

Prepare the dataset for training by creating a **validation split** and converting all sequences to a **fixed length**.

### Student Tasks

- Create a **validation set** from the training data (`VAL_SIZE = 5000`).  
  Use a **deterministic split from the end of the training set**.

- Define a maximum sequence length **`MAX_LEN`** (e.g., 200–300 tokens) based on the sequence statistics observed in **Q1**.

- Apply **sequence padding and truncation** using `pad_sequences` so that all reviews have the same length:
  - Use **post-padding**
  - Use **post-truncation**

- Generate padded datasets for:
  - `x_train_pad`
  - `x_val_pad`
  - `x_test_pad`

- Keep it **consistent across all models**.

- Print the shapes of the padded arrays to confirm the preprocessing step completed successfully.

---

In [3]:
# ============================================================
# Question Q2 — Validation Split & Sequence Padding (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define MAX_LEN and validation size
# 2) Create a validation split from the training data
# 3) Pad/truncate sequences to a fixed length
# 4) Verify resulting dataset shapes
# ============================================================

from tensorflow.keras.preprocessing.sequence import pad_sequences

# TODO 1: Define sequence length and validation size
MAX_LEN = 250
VAL_SIZE = 5000

# TODO 2: Create validation split from the end of the training set
x_val, y_val = x_train[-VAL_SIZE:], y_train[-VAL_SIZE:]
x_train2, y_train2 = x_train[:-VAL_SIZE], y_train[:-VAL_SIZE]

# TODO 3: Apply padding and truncation
x_train_pad = pad_sequences(
    x_train2,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_val_pad = pad_sequences(
    x_val,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

x_test_pad = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post",
)

# Print dataset shapes
print("x_train_pad:", x_train_pad.shape)
print("x_val_pad:  ", x_val_pad.shape)
print("x_test_pad: ", x_test_pad.shape)

x_train_pad: (20000, 250)
x_val_pad:   (5000, 250)
x_test_pad:  (25000, 250)


---

## Q3 — Build `tf.data` Pipelines

Create efficient **data pipelines** for training, validation, and testing using **TensorFlow `tf.data`**.

### Student Tasks

- Convert the padded datasets into **TensorFlow datasets** using `tf.data.Dataset.from_tensor_slices`.

- Create datasets for:
  - **training**
  - **validation**
  - **testing**

- Apply the following pipeline steps:
  - **shuffle** the training dataset
  - **batch** the datasets using an appropriate batch size
  - use **prefetching** to improve training performance

- Ensure the pipelines are ready to be used directly in **model training with `model.fit()`**.


---

In [4]:
# ============================================================
# Question Q3 — tf.data Pipelines (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define batch size and AUTOTUNE
# 2) Create TensorFlow datasets from the padded arrays
# 3) Apply shuffle, batch, and prefetch operations
# 4) Prepare pipelines for training, validation, and testing
# ============================================================

# TODO 1: Define batch size and autotune
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# TODO 2: Create training dataset
train_ds = tf.data.Dataset.from_tensor_slices((x_train_pad, y_train2))

# TODO 3: Apply shuffle, batch, and prefetch
train_ds = train_ds.shuffle(len(x_train_pad), seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 4: Create validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((x_val_pad, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# TODO 5: Create test dataset
test_ds = tf.data.Dataset.from_tensor_slices((x_test_pad, y_test))
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Pipelines ready.")

Pipelines ready.


---

## Q4 — Define Training Utilities

In this step, you will create **reusable helper functions** that simplify model training, evaluation, and experiment management.

These utilities will be used for **all models later in the assignment**.

### Student Tasks

1. **Create training callbacks**

   Define a function that returns commonly used training callbacks, including:

   - **EarlyStopping** to stop training when validation performance stops improving.
   - **ReduceLROnPlateau** to automatically reduce the learning rate when validation loss plateaus.
   - **ModelCheckpoint** to save the **best-performing model** during training.

2. **Define a model compilation function**

   Implement a function that compiles a model using:

   - **Adam optimizer**
   - **Binary cross-entropy loss** for sentiment classification
   - **Accuracy** as the evaluation metric

3. **Create a training function**

   Implement a function that trains a model using:

   - the **training dataset**
   - the **validation dataset**
   - the callbacks defined above

4. **Create an evaluation function**

   Implement a function that evaluates a trained model on a dataset and reports:

   - **loss**
   - **accuracy**

These functions will help keep the notebook **organized, reusable, and consistent across experiments**.

---

In [5]:
# ============================================================
# Question Q4 — Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define callbacks for training
# 2) Compile the model with optimizer, loss, and metrics
# 3) Train the model using training and validation datasets
# 4) Evaluate the trained model on a dataset
# ============================================================

# TODO 1: Define training callbacks
def build_callbacks(run_name: str):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.25, patience=1),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f"{run_name}.keras",
            monitor="val_accuracy",
            save_best_only=True,
        ),
    ]


# TODO 2: Compile model
def compile_model(model, lr=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
        metrics=["accuracy"],
    )
    return model


# TODO 3: Train model
def train_model(model, run_name: str, epochs=8):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=build_callbacks(run_name),
        verbose=1,
    )
    return history


# TODO 4: Evaluate model
def evaluate_model(model, name: str, ds):
    loss, acc = model.evaluate(ds, verbose=0)
    print(f"{name}: loss={loss:.4f}, acc={acc:.4f}")
    return loss, acc

---

## Q5 — Model A: Vanilla RNN

Build a **Vanilla RNN** model for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add a **SimpleRNN layer** for sequence processing.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.

---

In [6]:
# ============================================================
# Question Q5 — Model A: Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define embedding dimension and RNN units
# 2) Build a Sequential RNN model
# 3) Add Input, Embedding, RNN, and Dense layers
# 4) Compile the model
# 5) Display the model summary
# ============================================================

from tensorflow.keras import layers

# TODO 1: Define model hyperparameters (e.g., 128, 128)
EMBED_DIM = 128
RNN_UNITS = 128
maxInputSize = MAX_LEN
# TODO 2: Build the Sequential model
rnn_model = tf.keras.Sequential([

    # TODO 3: Input layer
    layers.Input(shape=(maxInputSize,)),

    # TODO 4: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE + 1 , output_dim=EMBED_DIM, mask_zero = True),

    # TODO 5: Simple RNN layer
    layers.SimpleRNN(RNN_UNITS),

    # TODO 6: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="vanilla_rnn")

# TODO 7: Compile the model
rnn_model = compile_model(rnn_model, lr=1e-3)

# Print model summary
rnn_model.summary()

Model: "vanilla_rnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 250, 128)       │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,153 (5.01 MB)

 Trainable params: 1,313,153 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# ============================================================
# Train and Evaluate Vanilla RNN (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the Vanilla RNN model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the RNN model
history_rnn = train_model(rnn_model, "proj5_vanilla_rnn", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(rnn_model, "Validation (RNN)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(rnn_model, "Test (RNN)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 32ms/step - accuracy: 0.5557 - loss: 0.6799 - val_accuracy: 0.5970 - val_loss: 0.6584 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.6879 - loss: 0.5828 - val_accuracy: 0.6834 - val_loss: 0.5951 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.7931 - loss: 0.4514 - val_accuracy: 0.7922 - val_loss: 0.4806 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.8235 - loss: 0.4115 - val_accuracy: 0.5508 - val_loss: 0.7196 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.8425 - loss: 0.3668 - val_accuracy: 0.7544 - val_loss: 0.5305 - learning_rate: 2.5000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.9104 - loss: 0.2398 - val_accuracy: 0.8016 - val_loss: 0.4855 - learning_rate: 6.2500e-05
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.9190 - loss: 

(0.48318979144096375, 0.8004000186920166)

---

## Q6 — Model B: LSTM

Build an **LSTM-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add an **Embedding layer** for word representations.
- Add an **LSTM layer** to capture long-term dependencies in sequences.
- Add a **Dense output layer** with **sigmoid activation**.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [8]:
from re import VERBOSE
# ============================================================
# Question Q6 — Model B: LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build an LSTM-based sequence model
# 2) Add Input, Embedding, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential LSTM model
lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(maxInputSize,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE + 1 , output_dim=EMBED_DIM, mask_zero = True),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm")

# TODO 6: Compile the model
lstm_model = compile_model(lstm_model, lr=1e-3)

# Print model summary
lstm_model.summary()

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 250, 128)       │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,411,841 (5.39 MB)

 Trainable params: 1,411,841 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ============================================================
# Train and Evaluate LSTM (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the LSTM model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the LSTM model
history_lstm = train_model(lstm_model, run_name="proj5_lstm", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(lstm_model, "Validation (LSTM)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(lstm_model, "Test (LSTM)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.7750 - loss: 0.4686 - val_accuracy: 0.8520 - val_loss: 0.3566 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8672 - loss: 0.3269 - val_accuracy: 0.8490 - val_loss: 0.3644 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9216 - loss: 0.2095 - val_accuracy: 0.8616 - val_loss: 0.3376 - learning_rate: 2.5000e-04
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.9416 - loss: 0.1655 - val_accuracy: 0.8632 - val_loss: 0.3432 - learning_rate: 2.5000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9593 - loss: 0.1266 - val_accuracy: 0.8666 - val_loss: 0.3737 - learning_rate: 6.2500e-05
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.9630 - loss: 0.1157 - val_accuracy: 0.8664 - val_loss: 0.3728 - learning_rate: 1.5625e-05
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.9646 

(0.40279620885849, 0.8593599796295166)

---

## Q7 — Model C: GRU

Build a **GRU-based model** for sentiment classification.

### Student Tasks

- Create a **Sequential model**.
- Add **Embedding → GRU → Dense(sigmoid)** layers.
- **Compile the model** using the training utility function.
- Print the **model summary**.


---

In [10]:
# ============================================================
# Question Q7 — Model C: GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a GRU-based sequence model
# 2) Add Input, Embedding, GRU, and Dense layers
# 3) Compile the model using the training utility
# 4) Display the model summary
# ============================================================

# TODO 1: Build the Sequential GRU model
gru_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(maxInputSize,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE + 1 , output_dim=EMBED_DIM, mask_zero = True),

    # TODO 4: GRU layer
    layers.GRU(RNN_UNITS),

    # TODO 5: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="gru")

# TODO 6: Compile the model
gru_model = compile_model(gru_model, lr=1e-3)

# Print model summary
gru_model.summary()

Model: "gru"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 250, 128)       │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,379,329 (5.26 MB)

 Trainable params: 1,379,329 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# ============================================================
# Train and Evaluate GRU (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Train the GRU model
# 2) Evaluate the model on the validation dataset
# 3) Evaluate the model on the test dataset
# ============================================================

# TODO 1: Train the GRU model
history_gru = train_model(gru_model, run_name="proj5_gru", epochs=8)

# TODO 2: Evaluate on validation set
evaluate_model(gru_model, "Validation (GRU)", val_ds)

# TODO 3: Evaluate on test set
evaluate_model(gru_model, "Test (GRU)", test_ds)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.7444 - loss: 0.4915 - val_accuracy: 0.8612 - val_loss: 0.3323 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8852 - loss: 0.2853 - val_accuracy: 0.8666 - val_loss: 0.3288 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.9286 - loss: 0.1926 - val_accuracy: 0.8600 - val_loss: 0.3391 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9669 - loss: 0.1007 - val_accuracy: 0.8746 - val_loss: 0.3624 - learning_rate: 2.5000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9794 - loss: 0.0700 - val_accuracy: 0.8794 - val_loss: 0.3754 - learning_rate: 6.2500e-05
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9819 - loss: 0.0626 - val_accuracy: 0.8776 - val_loss: 0.3800 - learning_rate: 1.5625e-05
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9830 - los

(0.4229203760623932, 0.8632400035858154)

---

## 8) Complex Model D — Stacked LSTM with Dropout

In this question, build a **deeper LSTM-based sequence classifier** using two recurrent layers.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `Dropout(...)`
  - `LSTM(...)`
  - `Dense(1, activation="sigmoid")`
- Train the model using the same optimizer and callbacks.
- Evaluate on validation and test sets.
- Compare it with the single-layer LSTM from Q6.

### Goal
Study whether **stacking recurrent layers** helps the model learn richer sequential sentiment patterns.


---

In [12]:
# ============================================================
# Question Q8 — Complex Model D: Stacked LSTM + Dropout (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a stacked LSTM model with dropout
# 2) Add Input, Embedding, LSTM, Dropout, LSTM, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential stacked LSTM model
stacked_lstm_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(maxInputSize,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE + 1 , output_dim=EMBED_DIM, mask_zero = True),

    # TODO 4: First LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: Dropout layer
    layers.Dropout(0.2),

    # TODO 6: Second LSTM layer
    layers.LSTM(RNN_UNITS // 2),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="stacked_lstm_dropout")

# TODO 8: Compile the model
stacked_lstm_model = compile_model(stacked_lstm_model, lr=1e-3)

# Print model summary
stacked_lstm_model.summary()

# TODO 9: Train the model
history_stacked_lstm = train_model(
    stacked_lstm_model,
    run_name="proj5_stacked_lstm_dropout",
    epochs=8
)

# TODO 10: Evaluate on validation set
evaluate_model(stacked_lstm_model, "Validation (Stacked LSTM + Dropout)", val_ds)

# TODO 11: Evaluate on test set
evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)

Model: "stacked_lstm_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 250, 128)       │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 250, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,185 (5.57 MB)

 Trainable params: 1,461,185 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.7193 - loss: 0.5402 - val_accuracy: 0.5968 - val_loss: 0.6506 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.7391 - loss: 0.5072 - val_accuracy: 0.8384 - val_loss: 0.3867 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.8892 - loss: 0.2851 - val_accuracy: 0.8670 - val_loss: 0.3368 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.9238 - loss: 0.2103 - val_accuracy: 0.8598 - val_loss: 0.3761 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.9535 - loss: 0.1419 - val_accuracy: 0.8636 - val_loss: 0.3876 - learning_rate: 2.5000e-04
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.9668 - loss: 0.1109 - val_accuracy: 0.8628 - val_loss: 0.3815 - learning_rate: 6.2500e-05
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9708 - loss:

(0.4513375759124756, 0.8479200005531311)

---

## 9) Complex Model E — Encoder–Decoder LSTM Classifier

In this question, build an **encoder–decoder style recurrent model** for sentiment classification.

### Idea
- The **encoder LSTM** reads the review and produces a compact context representation.
- A `RepeatVector` creates a short decoded sequence from that context.
- A **decoder LSTM** transforms the context into a richer hidden representation.
- A final dense layer predicts the review sentiment.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM` encoder
  - `RepeatVector(1)`
  - `LSTM` decoder
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it with the simpler one-layer LSTM.

### Goal
Explore whether a **more structured encoder–decoder design** is useful for sequence classification, even though it is more common in seq2seq tasks.


---

In [13]:
# ============================================================
# Question Q9 — Complex Model E: Encoder-Decoder LSTM Classifier (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define the input layer
# 2) Add Embedding, Encoder LSTM, RepeatVector, Decoder LSTM, and Dense layers
# 3) Build the functional model
# 4) Compile the model using the training utility
# 5) Train the model
# 6) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Define input layer
encdec_inputs = layers.Input(shape=(maxInputSize,))

# TODO 2: Embedding layer
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(encdec_inputs)

# TODO 3: Encoder LSTM
encoded = layers.LSTM(RNN_UNITS)(x)

# TODO 4: Repeat encoded representation
decoded = layers.RepeatVector(1)(encoded)

# TODO 5: Decoder LSTM
decoded = layers.LSTM(RNN_UNITS // 2)(decoded)

# TODO 6: Output layer
encdec_outputs = layers.Dense(1, activation="sigmoid")(decoded)

# TODO 7: Build functional model
encdec_model = tf.keras.Model(encdec_inputs, encdec_outputs, name="encdec_lstm_classifier")

# TODO 8: Compile the model
encdec_model = compile_model(encdec_model, lr=1e-3)

# TODO 9: Print model summary
encdec_model.summary()

# TODO 10: Train the model
history_encdec = train_model(
    encdec_model,
    run_name="proj5_encdec_lstm",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(encdec_model, "Validation (Encoder-Decoder LSTM)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)

Model: "encdec_lstm_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 250)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 250, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,057 (5.57 MB)

 Trainable params: 1,461,057 (5.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.5035 - loss: 0.6932 - val_accuracy: 0.5068 - val_loss: 0.6926 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.5522 - loss: 0.6748 - val_accuracy: 0.5552 - val_loss: 0.6723 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.5882 - loss: 0.6278 - val_accuracy: 0.5812 - val_loss: 0.6663 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.7631 - loss: 0.4798 - val_accuracy: 0.7732 - val_loss: 0.5210 - learning_rate: 0.0010
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8833 - loss: 0.2948 - val_accuracy: 0.8304 - val_loss: 0.4125 - learning_rate: 0.0010
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9341 - loss: 0.1851 - val_accuracy: 0.8434 - val_loss: 0.4285 - learning_rate: 0.0010
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.9703 - loss: 0.0992 - 

(0.5371934175491333, 0.8339999914169312)

---

## 10) Complex Model F — LSTM + GRU Hybrid

In this question, build a **hybrid recurrent architecture** that combines LSTM and GRU layers without using bidirectional processing.

### Tasks
- Use the architecture:
  - `Embedding`
  - `LSTM(..., return_sequences=True)`
  - `GRU(...)`
  - `Dropout`
  - `Dense(1, activation="sigmoid")`
- Train and evaluate the model.
- Compare it against all earlier models in terms of accuracy and complexity.

### Goal
Test whether combining **LSTM-based memory** with a **GRU-based final sequence encoder** captures richer sentiment patterns than a single recurrent layer.


---

In [14]:
# ============================================================
# Question Q10 — Complex Model F: LSTM + GRU Hybrid (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a hybrid sequence model using LSTM and GRU layers
# 2) Add Input, Embedding, LSTM, GRU, Dropout, and Dense layers
# 3) Compile the model using the training utility
# 4) Train the model
# 5) Evaluate the model on validation and test datasets
# ============================================================

# TODO 1: Build the Sequential hybrid model
hybrid_model = tf.keras.Sequential([

    # TODO 2: Input layer
    layers.Input(shape=(maxInputSize,)),

    # TODO 3: Embedding layer
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim= EMBED_DIM, mask_zero= True),

    # TODO 4: LSTM layer
    layers.LSTM(RNN_UNITS, return_sequences=True),

    # TODO 5: GRU layer
    layers.GRU(RNN_UNITS // 2, dropout=-.2, recurrent_dropout=0.2),

    # TODO 6: Dropout layer
    layers.Dropout(0.2),

    # TODO 7: Output layer
    layers.Dense(1, activation="sigmoid"),

], name="lstm_gru_hybrid")

# TODO 8: Compile the model
hybrid_model = compile_model(hybrid_model, lr=1e-3)

# TODO 9: Print model summary
hybrid_model.summary()

# TODO 10: Train the model
history_hybrid = train_model(
    hybrid_model,
    run_name="proj5_lstm_gru_hybrid",
    epochs=8
)

# TODO 11: Evaluate on validation set
evaluate_model(hybrid_model, "Validation (LSTM + GRU Hybrid)", val_ds)

# TODO 12: Evaluate on test set
evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

Model: "lstm_gru_hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 250, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 250, 128)       │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,448,897 (5.53 MB)

 Trainable params: 1,448,897 (5.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 289s 912ms/step - accuracy: 0.7667 - loss: 0.4743 - val_accuracy: 0.8246 - val_loss: 0.4106 - learning_rate: 0.0010
Epoch 2/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 318s 900ms/step - accuracy: 0.8834 - loss: 0.2947 - val_accuracy: 0.8626 - val_loss: 0.3411 - learning_rate: 0.0010
Epoch 3/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 281s 897ms/step - accuracy: 0.9198 - loss: 0.2166 - val_accuracy: 0.8610 - val_loss: 0.3895 - learning_rate: 0.0010
Epoch 4/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 285s 911ms/step - accuracy: 0.9616 - loss: 0.1141 - val_accuracy: 0.8592 - val_loss: 0.4358 - learning_rate: 2.5000e-04
Epoch 5/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 318s 899ms/step - accuracy: 0.9801 - loss: 0.0702 - val_accuracy: 0.8608 - val_loss: 0.4595 - learning_rate: 6.2500e-05
Epoch 6/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 279s 893ms/step - accuracy: 0.9826 - loss: 0.0622 - val_accuracy: 0.8588 - val_loss: 0.4630 - learning_rate: 1.5625e-05
Epoch 7/8
313/313 ━━━━━━━━━━━━━━━━━━━━ 280s 894ms/step - a

(0.3555346131324768, 0.8520799875259399)

---

## Q11 — Performance Comparison Table

Create a compact table comparing all completed models:

- Vanilla RNN
- LSTM
- GRU
- Stacked LSTM + Dropout
- Encoder–Decoder LSTM
- LSTM + GRU Hybrid

### Student Tasks
- Create a comparison table summarizing the results of all models.
- Include validation accuracy and test accuracy.
- Identify the best-performing recurrent model.
- Briefly comment on whether more complex architectures were helpful.


---

In [15]:
# ============================================================
# Question Q11 — Performance Comparison Table (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Define helper functions for validation and training accuracy
# 2) Evaluate all trained models on the test dataset
# 3) Store model names and metrics in summary rows
# 4) Create a pandas DataFrame for comparison
# 5) Sort the table by test accuracy
# ============================================================

import pandas as pd

# TODO 1: Best validation accuracy from training history
def best_val_acc(history):
    return max(history.history.get("val_accuracy", [np.nan]))

# TODO 2: Final training accuracy from training history
def final_train_acc(history):
    return history.history.get("accuracy", [np.nan])[-1]

summary_rows = []

# TODO 3: Evaluate all models on the test set
_, rnn_test_acc = evaluate_model(rnn_model, "Test (Vanilla RNN)", test_ds)
_, lstm_test_acc = evaluate_model(lstm_model, "Test (LSTM)", test_ds)
_, gru_test_acc = evaluate_model(gru_model, "Test (GRU)", test_ds)
_, stacked_lstm_test_acc = evaluate_model(stacked_lstm_model, "Test (Stacked LSTM + Dropout)", test_ds)
_, encdec_test_acc = evaluate_model(encdec_model, "Test (Encoder-Decoder LSTM)", test_ds)
_, hybrid_test_acc = evaluate_model(hybrid_model, "Test (LSTM + GRU Hybrid)", test_ds)

# TODO 4: Append summary rows
summary_rows.append(["Vanilla RNN", final_train_acc(history_rnn), best_val_acc(history_rnn), rnn_test_acc])
summary_rows.append(["LSTM", final_train_acc(history_lstm), best_val_acc(history_lstm), lstm_test_acc])
summary_rows.append(["GRU", final_train_acc(history_gru), best_val_acc(history_gru), gru_test_acc])
summary_rows.append(["Stacked LSTM + Dropout", final_train_acc(history_stacked_lstm), best_val_acc(history_stacked_lstm), stacked_lstm_test_acc])
summary_rows.append(["Encoder-Decoder LSTM", final_train_acc(history_encdec), best_val_acc(history_encdec), encdec_test_acc])
summary_rows.append(["LSTM + GRU Hybrid", final_train_acc(history_hybrid), best_val_acc(history_hybrid), hybrid_test_acc])

# TODO 5: Create DataFrame
results_df = pd.DataFrame(
    summary_rows,
    columns=["Model", "Final Train Acc", "Best Val Acc", "Test Acc"]
)

# TODO 6: Sort by test accuracy
results_df = results_df.sort_values(by="Test Acc", ascending=False).reset_index(drop=True)

# Display results
results_df

Test (Vanilla RNN): loss=0.4832, acc=0.8004
Test (LSTM): loss=0.4028, acc=0.8594
Test (GRU): loss=0.4229, acc=0.8632
Test (Stacked LSTM + Dropout): loss=0.4513, acc=0.8479
Test (Encoder-Decoder LSTM): loss=0.5372, acc=0.8340
Test (LSTM + GRU Hybrid): loss=0.3555, acc=0.8521


,Model,Final Train Acc,Best Val Acc,Test Acc
0,GRU,0.98295,0.8794,0.86324
1,LSTM,0.96495,0.8668,0.85936
2,LSTM + GRU Hybrid,0.98375,0.8626,0.85208
3,Stacked LSTM + Dropout,0.97065,0.8672,0.84792
4,Encoder-Decoder LSTM,0.98055,0.8460,0.83400
5,Vanilla RNN,0.92080,0.8034,0.80040


---

# Results and Discussions

Use the results above to answer the discussion questions below. You may revise the answers based on your actual experimental results (**Q1-Q11**).

---

## **Q12** — Which model performed best overall, and why might it outperform the others?  

The model that performs the best overall is the GRU model. It has the highest validation accuracy as well as test accuracy. The GRU overall is a simpler and faster version of an LSTM-based model, so the main difference is that the LSTM is still worse than the GRU model, which may be due to the amount of training data. The LSTM may provide a better estimate for data with more complex data, and with that, the hybrid LSTM and GRU model will always be a mix of the two, taking potential benefits from both, but our dataset shows no benefit above a non-hybrid version.

---

## **Q13** — Why does a vanilla RNN usually struggle more on long reviews?  

A vanilla RNN usually struggles with long reviews because the information needs to pass through many time steps, and the gradient can vanish or explode during training. During this process, the model may forget important earlier words. This makes it harder to learn long-term dependencies in the text. For sentiment classification, the key opinion may appear early in the review, but the RNN may not remember it very well at the end.  Compared with GRU or LSTM, vanilla RNN does not have gate mechanisms, so it is weaker at keeping useful information for a long sequence.

---

## **Q14** — Why might a more complex model not always outperform a simpler GRU or LSTM?  

A more complex model does not always perform better because a more complex model usually has extra layers, and hybrid designs increase the number of parameters and make optimization harder. In this task (IMDB Movie Reviews), a simpler GRU or LSTM may already be strong enough to capture the important information in the reviews, so adding more depth can lead to overfitting, slower training, and perform worse on validation or test data.
---

## **Q15** — What is the main idea behind the encoder–decoder classifier used here?  

The main objective is to evaluate whether an encoder–decoder architecture can improve performance in classification tasks. Compared to the plain LSTM model, the encoder compresses the sequential information into a single vector, forcing the model to store the most important information about the review and remove the noise. For sentiment, that may help the model focus on overall polarity but may also remove useful information.

Commonly, the encoder–decoder classifier is used for seq2seq tasks like translation, where the model must generate outputs step-by-step, so it relies on a decoder that uses both the encoded context and previously generated words. In contrast, classification tasks require only a single output, so the model benefits more from preserving rich information from the entire input sequence–such as stacked LSTMs– than from compressing it into a context and decoding it.


---

### 🎉 Congratulations!

You have successfully completed **A5 — Sequence Modeling with RNNs**. Excellent work applying and comparing **RNN, LSTM, and GRU architectures** for a **sequence-based text classification** task using the **IMDB Movie Reviews dataset**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q11)** have been executed successfully
- All **Markdown responses (Q12–Q15)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.